![Week 5 Day 4](../../assets/banners/w5d4.png)

# Week 5 Day 4 - Solution: A Smarter Reviewer

**The exercise:** write `triage_reviewer(payload)` for `grade_batch(..., reviewer=...)`: approve the safe cases automatically, and leave anything involving injections, illegible text or very low transcription confidence for Asha.

Try the exercise yourself first: struggling a little is where the learning happens! Then compare with this one. Everything runs offline with stand-ins (🎭); the 🔑 cells use your real models.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("❌ OpenAI API Key not set. Most of today's cells still run offline; the few that call a model will")
    print("   show a friendly 'Missing API key' panel. See Part 5 of the setup guide, or setup/SWAP_PROVIDER.md")

In [ ]:
import json

from langchain_core.embeddings import DeterministicFakeEmbedding

from gradewise.config import DATA_DIR, OUTPUT_DIR, settings
from gradewise.console import success
from gradewise.evals.compat import import_pandas
from gradewise.ingestion import ingest
from gradewise.rag import get_retriever
from gradewise.schemas import AnswerSheet, ExamPaper

pd = import_pandas()  # 💡 = `import pandas as pd`, with a fix for locked-down Windows machines

EXAM_FILE = DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json"
exam = ExamPaper.model_validate_json(EXAM_FILE.read_text(encoding="utf-8"))
QUESTIONS = {q.id: q for q in exam.questions}
TRUTH = json.loads((DATA_DIR / "answer-sheets" / "synthetic" / "transcripts.json").read_text(encoding="utf-8"))

# 🆓 A keyword-only (BM25) copy of the CS201 notes: fake embeddings, real keyword search, zero cost
OFFLINE_DB = OUTPUT_DIR / "week5_offline_db"
fake_embeddings = DeterministicFakeEmbedding(size=64)
ingest(collection="cs201_offline", embeddings=fake_embeddings, persist_dir=OFFLINE_DB, quiet=True)
bm25 = get_retriever("cs201_offline", k=4, mode="bm25", embeddings=fake_embeddings, persist_dir=OFFLINE_DB)

success(
    "Ready",
    {
        "Exam": f"{exam.title} ({len(exam.questions)} questions, {exam.total_marks:g} marks)",
        "Students": f"{len(TRUTH)} synthetic scripts",
        "Offline KB": "cs201_offline (BM25)",
        "Grading model": settings.grading_model,
    },
)

In [ ]:
REVIEW_LOG: list[dict] = []
# 💡 "low transcription confidence" is safe HERE: any such answer already passed the review_transcription
#    interrupt (auto-approved at >= 0.6 below, or checked by Asha) before it was graded.
AUTO_OK = ("low grader confidence", "low transcription confidence")
MUST_SEE = ("prompt injection", "illegible", "moderator disagreed")
MIN_TRANSCRIPTION_CONFIDENCE = 0.6


def triage_reviewer(payload: dict) -> dict | None:
    """Answer an orchestrator interrupt automatically when it's safe; return None to wait for Asha."""
    sid, kind = payload["student_id"], payload["type"]
    if kind == "review_transcription":
        worst = min(item["confidence"] for item in payload["items"])
        if worst >= MIN_TRANSCRIPTION_CONFIDENCE:
            REVIEW_LOG.append(
                {"student": sid, "type": kind, "decision": "auto-approved", "why": f"lowest confidence {worst:.2f}"}
            )
            return {"approved": True, "corrections": {}}
        REVIEW_LOG.append({"student": sid, "type": kind, "decision": "wait for Asha", "why": f"confidence {worst:.2f}"})
        return None
    reasons = [item["reason"] or "" for item in payload["items"]]
    if any(word in r for r in reasons for word in MUST_SEE):
        REVIEW_LOG.append(
            {
                "student": sid,
                "type": kind,
                "decision": "wait for Asha",
                "why": next(r for r in reasons if any(w in r for w in MUST_SEE))[:70],
            }
        )
        return None
    # 💡 Every reason must be a known-safe one: anything unexpected also waits for a human.
    if all(r and all(part.strip().startswith(AUTO_OK) for part in r.split(";")) for r in reasons):
        REVIEW_LOG.append(
            {"student": sid, "type": kind, "decision": "auto-approved", "why": f"{len(reasons)} low-confidence marks"}
        )
        return {"approved": True, "overrides": {}}
    REVIEW_LOG.append({"student": sid, "type": kind, "decision": "wait for Asha", "why": "unrecognised reason"})
    return None

A tough test: a stand-in grader with **low confidence** (0.65, below the 0.7 review threshold), so *every* answer is flagged, plus two shaky transcriptions: student 03's Q2a at 0.65 (fixable) and student 07's Q3 at 0.42 (not). One batch per student, so we can count how many still need Asha.

In [ ]:
from datetime import datetime

from gradewise.grading.fakes import KeywordGraderModel
from gradewise.orchestrator import build_orchestrator, grade_batch, sheets_from_transcripts

SANDBOX = OUTPUT_DIR / "week5" / "day4_solution"
app = build_orchestrator(
    "memory",
    llm=KeywordGraderModel(confidence=0.65),
    retriever=bm25,
    feedback="rules",
    pdf=False,
    index=False,
    db=SANDBOX / "gradewise.db",
    reports_dir=SANDBOX / "reports",
)  # 🎭 stand-in


def shaky(sheet: AnswerSheet, question_id: str, confidence: float) -> AnswerSheet:
    answers = [
        a.model_copy(update={"confidence": confidence}) if a.question_id == question_id else a for a in sheet.answers
    ]
    return sheet.model_copy(update={"answers": answers})


sheets = {s.student_id: s for s in sheets_from_transcripts(exam)}
sheets["student_03"] = shaky(sheets["student_03"], "Q2a", 0.65)
sheets["student_07"] = shaky(sheets["student_07"], "Q3", 0.42)

stamp = f"{datetime.now():%H%M%S}"
outcomes = {
    sid: grade_batch(
        exam.exam_id, [sheet], thread_id=f"triage-{sid}-{stamp}", exam=exam, orchestrator=app, reviewer=triage_reviewer
    ).status
    for sid, sheet in sheets.items()
}
print(
    f"done automatically: {sum(s == 'done' for s in outcomes.values())} · waiting for Asha: "
    f"{[sid for sid, s in outcomes.items() if s == 'paused']}"
)
pd.DataFrame(REVIEW_LOG)

Eight of ten scripts go straight through; only the two that genuinely need a human wait: student 07 (the scan is too unclear to grade fairly) and student 10 (the injection attempt). That's the point of triage: Asha's attention goes where it matters.

⚠️ Two design choices worth defending:
- **Allow-list, not block-list.** A flag is auto-approved only if *every* reason is a known-safe one. A new kind of flag someone adds next month waits for a human by default.
- **Log every automatic decision.** An auto-approval is still a decision about a student's mark, so it must be auditable (in week 6 this log becomes part of the review screen).